<a href="https://colab.research.google.com/github/lucysiewers/New-Play/blob/main/New_Play.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#New Play
My app helps a gamer developer decide which game to make by which genre is most highly rated, which genre has few games but lots of players, and which genre has the most players that play for a long time.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/MIST 4600/


In [2]:
import pandas as pd
games = pd.read_csv(FOLDER + "steam_games.csv")

# Define a function to find the highest-rated genres
def highest_rated_genres (games):

  # Remove games with missing genres
  games = games.dropna(subset=["genres"]).copy()

  #Separate games that have multiple genres
  games ["genres"] = games ["genres"].str.split(",")
  games = games.explode("genres")

  #Remove extra space from genre names
  games ["genres"] = games["genres"].str.strip()

  # Calculate the average positive rating and number of games for each genre
  genre_ratings = games.groupby ("genres")["pct_pos_total"].agg(["mean", "count"])

  # Sort genres from highest to lowest rated
  genre_ratings = genre_ratings.sort_values("mean", ascending = False)

  #Return the top 10 highest-rated genres
  return genre_ratings.head(10)

#Call the function and display the results
print (highest_rated_genres(games))


                   mean  count
genres                        
Indie         89.648019    429
Casual        88.630137    146
Simulation    86.734375    256
Adventure     86.171765    425
Strategy      85.068293    205
Racing        84.967742     31
Action        84.406699    627
RPG           84.003717    269
Early Access  82.695652     69
Free To Play  78.597884    189


In [6]:
print(*games.columns.tolist(), sep='\n')

appid
name
release_date
release_year
price
discount
required_age
windows
mac
linux
developers
publishers
genres
n_languages
achievements
metacritic_score
positive
negative
pct_pos_total
num_reviews_total
estimated_owners
owners_low
avg_playtime_hours
peak_ccu
short_description
critic_reviews


In [9]:
# Create a copy of the dataset
games_2 = games.copy()

# Separate games with multiple genres
games_2["genres"] = games_2["genres"].str.split(",")
games_2 = games_2.explode("genres")

# Remove extra whitespace around genre names
games_2["genres"] = games_2["genres"].str.strip()

#Count games and calculate estimated owners by genre
genre_demand = games_2.groupby("genres").agg(number_of_games=("appid","nunique"),total_owners=("owners_low","sum"),average_owners=("owners_low","mean"))

#Sort by average number of owners per game
genre_demand = genre_demand.sort_values("average_owners",ascending=False)

#Show the top 10 genres
print(genre_demand.head(10))

#Find genres with fewer than 200 games
low_competition = genre_demand[genre_demand["number_of_games"] < 200]

#Sort by average estimated owners
low_competition = low_competition.sort_values("average_owners", ascending=False)

#Display the results
print(low_competition)


                       number_of_games  total_owners  average_owners
genres                                                              
Free To Play                       189     957800000    5.067725e+06
Massively Multiplayer              108     494600000    4.579630e+06
Action                             627    2379900000    3.795694e+06
RPG                                269     927400000    3.447584e+06
Adventure                          425    1385500000    3.260000e+06
Strategy                           205     616700000    3.008293e+06
Early Access                        69     195300000    2.830435e+06
Sports                              39     106200000    2.723077e+06
Indie                              429    1055200000    2.459674e+06
Casual                             146     349300000    2.392466e+06
                       number_of_games  total_owners  average_owners
genres                                                              
Free To Play                      

In [12]:
import requests

# CheapShark API URL
url = "https://www.cheapshark.com/api/1.0/games"

# Identify our application to CheapShark
headers = {"User-Agent": "SteamGameProject/1.0 (student project)"}

# Search for a specific game
parameters = {"title": "EA SPORTS FC 25", "limit": 5}

# Request information from CheapShark
response = requests.get(url, params=parameters, headers=headers)

# Convert the response to Python data
game_prices = response.json()

# Display the results vertically
for game in game_prices:
    print(f"Game: {game.get('external')}")
    print(f"Cheapest Price: ${game.get('cheapest')}")
    print(f"Steam App ID: {game.get('steamAppID')}")
    print("-" * 40)

Game: EA SPORTS FC 25 Ultimate Edition
Cheapest Price: $99.99
Steam App ID: None
----------------------------------------
Game: EA SPORTS FC 25 Standard Edition
Cheapest Price: $69.99
Steam App ID: None
----------------------------------------


In [13]:
#Find Sports games in the dataset
sports_games = games_2[games_2["genres"] == "Sports"]

#Sort by estimated number of owners
sports_games = sports_games.sort_values ("owners_low", ascending = False)

#Display the top 5 Sports games
print (sports_games[["name", "owners_low", "price"]].head(5))

                             name  owners_low  price
28                      Fall Guys    20000000   0.00
30                 Rocket League®    10000000   0.00
62                         VRChat    10000000   0.00
83                Forza Horizon 5     5000000  29.99
112  theHunter: Call of the Wild™     5000000  19.99


In [18]:
import requests

url = "https://www.cheapshark.com/api/1.0/games"
headers = {"User-Agent": "SteamGameProject/1.0 (student project)"}

#Search CheapShark for the top 5 Sports games
for game in sports_games["name"].head(5):
  parameters = {"title":game, "limit":20}
  response = requests.get(url, params=parameters, headers=headers, timeout=10)

  print("Steam Game:", game)

  if response.status_code == 200:
    game_prices = response.json()

    #Only use results matching the game's title
    matches = []

    for result in game_prices:
      clean_game = game.replace("®", "").replace("™", "").strip().lower()
      if result ["external"].strip().lower() == clean_game:
        matches.append(result)


    if len(matches) > 0:
      for result in matches:
        print("CheapShark Price: $", result["cheapest"])
    else:
      print("No exact base-game match found")

  else:
    print("API request failed")

  print("------------------------------")


Steam Game: Fall Guys
No exact base-game match found
------------------------------
Steam Game: Rocket League®
No exact base-game match found
------------------------------
Steam Game: VRChat
No exact base-game match found
------------------------------
Steam Game: Forza Horizon 5
CheapShark Price: $ 59.99
------------------------------
Steam Game: theHunter: Call of the Wild™
CheapShark Price: $ 3.00
------------------------------


In [22]:
import requests

def find_underserved_genres(games):
  #Separate games with multiple genres
  games_2 = games.copy()
  games_2["genres"] = games_2["genres"].str.split(",")
  games_2 = games_2.explode("genres")
  games_2["genres"] = games_2["genres"].str.strip()

  #Calculate games and owners for each genre
  genre_demand = games_2.groupby("genres").agg(
      number_of_games=("appid", "nunique"),
      total_owners=("owners_low", "sum"),
      average_owners=("owners_low", "mean")
  )

  #Find genres with fewer than 200 games
  low_competition = genre_demand[genre_demand["number_of_games"] < 200]
  low_competition = low_competition.sort_values(
      "average_owners", ascending=False
  )

  print("Genres with fewer games and high ownership:")
  print(low_competition)

  #Select the five most-owned sports games
  sports_games = games_2[games_2["genres"] == "Sports"]
  sports_games = sports_games.sort_values("owners_low", ascending=False)

  #Connect to CheapShark
  url = "https://www.cheapshark.com/api/1.0/games"
  headers = { "User-Agent": "SteamGameProject/1.0 (student project)" }

  print("\nCheapShark Pricing for Popular Sports Games:")

  for game in sports_games["name"].head(5):
    parameters = {"title": game, "limit": 20}

    try:
      response = requests.get(url, params=parameters, headers=headers, timeout=10)
      print("\nSteam Game:", game)

      if response.status_code == 200:
        game_prices = response.json()
        matches = []

        clean_game = game.replace("®", "").replace("™", "").strip().lower()

        for result in game_prices:
          if result["external"].strip().lower() == clean_game:
            matches.append(result)

        if len(matches) > 0:
          for result in matches:
            print("CheapShark Lowest Price: $", result["cheapest"])
        else:
          print("No exact base-game match found")
      else:
        print("API request failed")
    except requests.exceptions.RequestException:
      print("Could not connect to CheapShark")

  return low_competition

In [23]:
underserved_genres = find_underserved_genres(games)

Genres with fewer games and high ownership:
                       number_of_games  total_owners  average_owners
genres                                                              
Free To Play                       189     957800000    5.067725e+06
Massively Multiplayer              108     494600000    4.579630e+06
Early Access                        69     195300000    2.830435e+06
Sports                              39     106200000    2.723077e+06
Casual                             146     349300000    2.392466e+06
Racing                              31      59700000    1.925806e+06

CheapShark Pricing for Popular Sports Games:

Steam Game: Fall Guys
No exact base-game match found

Steam Game: Rocket League®
No exact base-game match found

Steam Game: VRChat
No exact base-game match found

Steam Game: Forza Horizon 5
CheapShark Lowest Price: $ 59.99

Steam Game: theHunter: Call of the Wild™
CheapShark Lowest Price: $ 3.00
